# X2: Evaluation and Benchmarking

A model that reports a good number and a model that actually works are not
the same claim. This notebook is about the gap between them: five specific,
common ways a measured result can be wrong even though the training run
itself was fine — leaked information, the wrong metric, an uncalibrated
probability, a single lucky seed, and a benchmark number that means less
than it appears to.

## Introduction

Every technique below shares one property: the training code runs without
error, the reported number looks plausible, and the mistake is invisible
unless the evaluation procedure itself is checked. That makes these among
the most expensive bugs in practice — they survive code review and produce
a number confident enough that nobody double-checks it.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, data
# generation, minibatch order) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from scipy import stats
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
device = torch.device("cpu")

## Splits and Leakage

The train/validation/test split exists so that the test score estimates
performance on genuinely unseen data: fit everything (including any
preprocessing or feature choice) on train, tune on validation, and touch
test exactly once, at the end. **Leakage** is any path by which information
from validation or test influences a choice made before that final check —
and one of the most common, subtle forms is selecting *which features to
use* by looking at the whole dataset, test included, before splitting.

To make the danger unmistakable, the data below is constructed so $y$ is
pure noise, entirely independent of $X$ — there is truly nothing to learn.

In [ ]:
N, D = 200, 1000
rng = np.random.default_rng(SEED)
X = rng.normal(size=(N, D))
y = rng.integers(0, 2, size=N)  # independent of X by construction: no true signal exists

N_TRAIN = 140
X_train_raw, X_test_raw = X[:N_TRAIN], X[N_TRAIN:]
y_train, y_test = y[:N_TRAIN], y[N_TRAIN:]


def top_k_by_correlation(Xs, ys, k=20):
    corr = np.array([np.corrcoef(Xs[:, j], ys)[0, 1] for j in range(Xs.shape[1])])
    return np.argsort(-np.abs(corr))[:k]


# Leaky: features selected by correlation with y computed over train+test combined,
# BEFORE the split is respected.
leaky_features = top_k_by_correlation(X, y, k=20)
clf_leaky = LogisticRegression(max_iter=1000).fit(X_train_raw[:, leaky_features], y_train)
acc_leaky = clf_leaky.score(X_test_raw[:, leaky_features], y_test)

# Correct: features selected using the train split only.
correct_features = top_k_by_correlation(X_train_raw, y_train, k=20)
clf_correct = LogisticRegression(max_iter=1000).fit(X_train_raw[:, correct_features], y_train)
acc_correct = clf_correct.score(X_test_raw[:, correct_features], y_test)

print(f"leaky procedure   -- test accuracy: {acc_leaky:.1%}  (features chosen using test labels too)")
print(f"correct procedure -- test accuracy: {acc_correct:.1%}  (features chosen from train only)")
print("true relationship between X and y: none -- y was generated independently of X")

On data with **zero true signal**, the leaky procedure still reports a
test accuracy well above chance, because feature selection was allowed to
see the test labels and could cherry-pick noise features that happen to
correlate with them too. The correct procedure, restricted to train-only
information, reports something close to the honest answer: chance. The
leaked test score is not slightly optimistic — it can indicate a working
classifier where none exists.

## Choosing Metrics

Accuracy is the wrong headline metric whenever classes are imbalanced: a
classifier that always predicts the majority class scores well on accuracy
while doing nothing useful at all. Precision, recall, F1 and ROC-AUC each
answer a different question, and the right one depends on what a mistake
costs — recall matters most when missing a positive is expensive (medical
screening), precision matters most when a false alarm is expensive (spam
filtering that deletes real mail), F1 balances both, and ROC-AUC scores
ranking quality independent of any chosen decision threshold.

In [ ]:
n_majority, n_minority = 180, 20
X_imb = np.concatenate([rng.normal(0, 1, (n_majority, 2)), rng.normal(2, 1, (n_minority, 2))])
y_imb = np.concatenate([np.zeros(n_majority), np.ones(n_minority)])
perm = rng.permutation(len(y_imb))
X_imb, y_imb = X_imb[perm], y_imb[perm]
n_tr = 150
Xtr, ytr, Xte, yte = X_imb[:n_tr], y_imb[:n_tr], X_imb[n_tr:], y_imb[n_tr:]

majority_preds = np.zeros_like(yte)  # always predicts class 0

real_clf = LogisticRegression().fit(Xtr, ytr)
real_preds = real_clf.predict(Xte)
real_probs = real_clf.predict_proba(Xte)[:, 1]

print(f"{'metric':<12}{'always-majority':>18}{'real classifier':>18}")
print(f"{'accuracy':<12}{(majority_preds == yte).mean():>18.1%}{(real_preds == yte).mean():>18.1%}")
print(f"{'precision':<12}{precision_score(yte, majority_preds, zero_division=0):>18.1%}{precision_score(yte, real_preds, zero_division=0):>18.1%}")
print(f"{'recall':<12}{recall_score(yte, majority_preds, zero_division=0):>18.1%}{recall_score(yte, real_preds, zero_division=0):>18.1%}")
print(f"{'F1':<12}{f1_score(yte, majority_preds, zero_division=0):>18.1%}{f1_score(yte, real_preds, zero_division=0):>18.1%}")
print(f"{'ROC-AUC':<12}{'undefined':>18}{roc_auc_score(yte, real_probs):>18.1%}")

The always-majority classifier's accuracy looks respectable — it matches
the class imbalance itself — but its recall and F1 are exactly zero: it
finds none of the minority class it exists to detect. Accuracy alone would
have hidden that completely.

## Calibration

A **calibrated** classifier's predicted probability matches the empirical
frequency of being correct: among every example it calls 70% likely
positive, roughly 70% actually should be positive. This is a separate
property from accuracy — a classifier can rank examples perfectly and still
be badly overconfident about its probabilities, which matters whenever a
downstream decision (a threshold, a risk estimate) trusts the number itself,
not just the ranking.

In [ ]:
def reliability_curve(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(probs, bin_edges[1:-1])
    confidences, accuracies = [], []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() > 0:
            confidences.append(probs[mask].mean())
            accuracies.append(labels[mask].mean())
    return np.array(confidences), np.array(accuracies)


well_calibrated_probs = real_clf.predict_proba(Xte)[:, 1]        # logistic regression directly optimises log-loss
overconfident_probs = well_calibrated_probs ** 3 / (well_calibrated_probs ** 3 + (1 - well_calibrated_probs) ** 3)

conf_good, acc_good = reliability_curve(well_calibrated_probs, yte)
conf_bad, acc_bad = reliability_curve(overconfident_probs, yte)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "k--", label="perfect calibration")
ax.plot(conf_good, acc_good, "o-", label="logistic regression")
ax.plot(conf_bad, acc_bad, "o-", label="artificially sharpened (overconfident)")
ax.set_xlabel("predicted probability"); ax.set_ylabel("empirical frequency of being correct")
ax.legend(); ax.set_title("Reliability diagram")
plt.show()

The well-calibrated curve sits close to the diagonal; the sharpened curve
sits below it in the high-confidence region — it claims near-certainty far
more often than it is actually right, which is exactly the pattern reported
for many modern deep networks (confident, but not calibrated) unless
corrected with a step like temperature scaling.

## Comparing Runs Properly

Two training configurations, compared by a single run each, can report
whichever one got the luckier seed. The fix is running each configuration
across several seeds and comparing the *distributions*, with a stated
method — here, an independent two-sample $t$-test on the resulting
accuracies — rather than eyeballing two single numbers.

In [ ]:
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 20, generator=g)
    w = torch.randn(20, 2, generator=g)
    y = (X @ w).argmax(dim=1)
    return X, y


X_all, y_all = make_data(300, 999)
X_tr, y_tr = X_all[:200], y_all[:200]
X_te, y_te = X_all[200:], y_all[200:]


def run_config(lr, seed):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 2))
    opt = optim.SGD(model.parameters(), lr=lr)
    for _ in range(40):
        loss = nn.functional.cross_entropy(model(X_tr), y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (model(X_te).argmax(1) == y_te).float().mean().item()


accs_A = [run_config(lr=0.03, seed=s) for s in range(8)]
accs_B = [run_config(lr=0.04, seed=s) for s in range(8)]

print("config A (lr=0.03):", [round(a, 3) for a in accs_A], f"mean={np.mean(accs_A):.3f}")
print("config B (lr=0.04):", [round(a, 3) for a in accs_B], f"mean={np.mean(accs_B):.3f}")
print(f"single-seed comparison, seed 3 (A) vs. seed 6 (B): A={accs_A[3]:.3f} > B={accs_B[6]:.3f} -- A looks better")

t_stat, p_value = stats.ttest_ind(accs_A, accs_B)
print(f"\nproper comparison -- independent two-sample t-test across all 8 seeds each:")
print(f"t={t_stat:.2f}, p={p_value:.4f} -- B is reliably better despite the single-seed comparison above")

fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot([accs_A, accs_B], tick_labels=["config A (lr=0.03)", "config B (lr=0.04)"])
ax.set_ylabel("test accuracy")
ax.set_title(f"8 seeds each -- t-test p={p_value:.4f}")
plt.show()

## Evaluating Language Models

11a defined perplexity as $\exp(\text{cross-entropy})$ — the model's average
per-token surprise. It is useful for comparing models *on the same test
set*, but it is not an absolute quality score: perplexity depends on the
test corpus's own difficulty just as much as on the model, so perplexities
measured on different corpora are not comparable at all.

In [ ]:
def unigram_perplexity(train_text, test_text):
    train_tokens = train_text.split()
    counts = {}
    for tok in train_tokens:
        counts[tok] = counts.get(tok, 0) + 1
    vocab_size = len(counts) + 1  # +1 for unseen-word smoothing
    total = sum(counts.values()) + vocab_size  # add-one (Laplace) smoothing

    test_tokens = test_text.split()
    log_probs = [np.log((counts.get(tok, 0) + 1) / total) for tok in test_tokens]
    return float(np.exp(-np.mean(log_probs)))


train_corpus = "the cat sat on the mat the dog ran in the park the cat and the dog played"
in_domain_test = "the cat sat on the mat in the park"          # same style and vocabulary as training
out_of_domain_test = "quantum entanglement violates local realism"  # entirely different domain

ppl_in_domain = unigram_perplexity(train_corpus, in_domain_test)
ppl_out_of_domain = unigram_perplexity(train_corpus, out_of_domain_test)
print(f"perplexity on in-domain test text:     {ppl_in_domain:.1f}")
print(f"perplexity on out-of-domain test text: {ppl_out_of_domain:.1f}")
print("same model, same training data -- the huge gap is entirely a property of the test text")

The same trained model reports wildly different perplexity depending only
on which text it is measured against — a lower number never proves a
'better' model unless the test set is held fixed. Two further weaknesses
compound this in practice: **benchmark contamination**, where evaluation
text leaks into pretraining data (the same leakage failure from "Splits and
Leakage", at corpus scale, discovered after the fact rather than by
design), and **automatic-metric misalignment**, where surface-overlap
metrics like BLEU and ROUGE correlate only weakly with human judgement of
open-ended generation quality — the reason human and LLM-as-judge
evaluation have become standard for exactly the tasks perplexity, BLEU and
ROUGE evaluate worst, despite carrying their own biases (favouring longer
or more confidently-phrased answers regardless of correctness).

## Key Takeaways

- **Leakage** through feature selection performed before splitting can turn
  pure noise into an apparently working classifier — measured here as 20+
  points of inflated test accuracy on data with zero true signal.
- **Accuracy hides failure on imbalanced classes**: a classifier that
  predicts nothing but the majority class scores well on accuracy while
  scoring exactly zero on recall and F1.
- **Calibration is separate from accuracy** — a reliability diagram exposes
  overconfidence that a single accuracy number cannot.
- **A single seed can report the wrong winner.** A proper comparison runs
  several seeds per configuration and applies a stated statistical test —
  demonstrated here by an explicit single-seed pair that favoured the
  reliably-worse configuration.
- **Perplexity is corpus-relative, not an absolute quality score**, and
  language-model benchmarks face further known weaknesses: contamination
  (test data leaking into pretraining) and automatic metrics that correlate
  only weakly with human judgement of open-ended generation.